# 3.10 AutoML : une recherche ne vaut que ce que vaut son score

🖥️ [**Diapositives du cours, Séance 18 (lun. 9 nov.)**](https://geo-smart.github.io/mlgeo-book/slides/2026/lec18_robust_training.html)

Les outils de *machine learning* automatisé (AutoML), tels que H2O AutoML, AutoGluon et auto-sklearn, automatisent la partie centrale d'un projet : essayer de nombreuses familles de modèles et de réglages, noter chacun, garder le meilleur, souvent empiler les gagnants. En dessous se trouve une boucle assez courte pour que vous l'écriviez vous-même : ce notebook Jupyter (carnet) n'a donc besoin que de scikit-learn. Les éditions antérieures de ce chapitre enseignaient pycaret et Optuna. Écrire la boucle à la main montre ce que ces paquets ne peuvent pas faire pour vous : ils classent les modèles selon le score que vous leur donnez, quel qu'il soit.

Lee et al. (2023, *Remote Sensing*) montrent pourquoi c'est important. L'AutoML a battu une forêt aléatoire pour l'humidité du sol journalière en Corée du Sud avec un échantillonnage aléatoire (erreur de 2,21 % contre 2,71 %). Lorsque des années entières étaient tenues à l'écart, les deux étaient à égalité (4,13 % contre 4,10 %).

**Objectifs d'apprentissage.** À la fin de ce notebook, vous saurez :

1. écrire un petit AutoML avec scikit-learn ;
2. montrer que le choix de la validation peut changer le gagnant, et mesurer ce que cela coûte sur un test final ;
3. dire pourquoi le meilleur score d'une recherche est optimiste ;
4. auditer un script de modélisation écrit par un agent d'IA, et le réparer.

**Durée :** environ 20 minutes, à deux. Si le temps manque, sautez la partie 2.

## Comment fonctionne ce notebook

Chaque cellule **À vous** demande quelques lignes de code. Lisez l'indice, écrivez votre réponse à l'endroit indiqué dans la cellule et exécutez-la. La cellule suivante contient une solution détaillée, masquée tant que vous ne cliquez pas sur ⋯. Cette cellule masquée vérifie aussi votre réponse : elle affiche `correct` ou un indice, et si votre réponse manque ou diffère, elle utilise le résultat de référence pour que la suite du notebook s'exécute quand même. Votre propre réponse, si elle diffère, est conservée sous le même nom suivi de `_yours`.

In [1]:
# Data handling
import hashlib
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import pooch

# Plotting
import matplotlib.pyplot as plt

# Models, splits and scores (scikit-learn)
from sklearn.datasets import make_classification   # synthetic stand-in if Zenodo cannot be reached
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import f1_score, mean_absolute_error
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

Les valeurs ci-dessous servent à plusieurs endroits : elles sont donc définies une seule fois. Les couleurs des figures restent distinctes pour les personnes daltoniennes (la palette Okabe-Ito).

In [2]:
SEED = 42
SPLIT_SEED = 2026   # the canonical split of notebook 3.5
BLUE, ORANGE, RED, MUTED = "#0072B2", "#E69F00", "#D55E00", "#6e675c"
plt.rcParams.update({"font.size": 12, "axes.spines.top": False, "axes.spines.right": False})

In [3]:
# Helper: compare an answer with a reference. If the answer is missing or different, print a hint and use the
# reference so the cells below still run; a different answer is kept as <name>_yours.
def check(name, ns, reference, hint, measure=None, tol=1e-3):
    measure = measure or (lambda value: np.asarray(value, dtype=float))
    yours = ns.get(name)
    if yours is None or (hasattr(yours, "__len__") and len(yours) == 0):
        print(f"{name}: not answered yet. Hint: {hint}\n  Using the reference result below.")
    else:
        try:
            ok = np.allclose(measure(yours), measure(reference), atol=tol)
        except Exception:
            ok = False
        if ok:
            print(f"{name}: correct")
            return
        print(f"{name}: not the expected result. Hint: {hint}\n  Your answer is kept as {name}_yours; "
              "the reference is used below.")
        ns[f"{name}_yours"] = yours
    ns[name] = reference

## Partie 1. Un petit AutoML, à la main

*Données :* 📍 Capteur in situ · 🪨 Terre solide · 📐 Géodésie

Nous reprenons la prévision de [3.8](3.8_robust_training.ipynb) : la station GNSS P395 dans la chaîne côtière de l'Oregon (Oregon Coast Range), 60 jours de position en composante est pour prédire la moyenne des 30 jours suivants. La recherche voit les premiers 80 % de l'enregistrement. Les derniers 20 % restent à l'écart pour un unique test final.

In [4]:
course_path = Path("../Chapter1-GettingStarted/data/gps_P395_relative_position.csv")
if not course_path.exists():
    course_path = Path("gps_P395_relative_position.csv")
    if not course_path.exists():
        urllib.request.urlretrieve("https://raw.githubusercontent.com/geo-smart/mlgeo-book/main/book/Chapter1-GettingStarted/data/gps_P395_relative_position.csv", course_path)
if hashlib.sha256(course_path.read_bytes()).hexdigest() != "eb22473bbc5565caafbfe1b3b539c65cf88ba3cb23dec07106d81738957a4ff6":
    raise ValueError(f"{course_path} is not the course copy. Delete it, or restore it with git, and run this cell again.")
gnss = pd.read_csv(course_path)
east = gnss["east_m"].to_numpy() * 1000   # mm, relative to the first day
year = gnss["decimal_year"].to_numpy()
history, horizon, stride = 60, 30, 5
rows = []
for i in range(history, len(east) - horizon, stride):
    past = east[i - history:i]
    rows.append({"mean_60d": past.mean(), "mean_10d": east[i - 10:i].mean(), "last": east[i - 1],
                 "slope_60d": np.polyfit(np.arange(history), past, 1)[0],
                 "season_sin": np.sin(2 * np.pi * year[i]), "season_cos": np.cos(2 * np.pi * year[i]),
                 "target": east[i:i + horizon].mean(), "year": year[i]})
samples = pd.DataFrame(rows)
FEATURES = ["mean_60d", "mean_10d", "last", "slope_60d", "season_sin", "season_cos"]
X_time, y_time = samples[FEATURES].to_numpy(), samples["target"].to_numpy()
last_value = samples["last"].to_numpy()
GAP = horizon // stride   # samples left out between training and validation, so target windows do not overlap
print(f"{len(samples)} samples, {X_time.shape[1]} features")

cut = int(0.8 * len(X_time))
X_search, y_search = X_time[:cut], y_time[:cut]
X_final, y_final = X_time[cut + GAP:], y_time[cut + GAP:]
print(f"{len(X_search)} samples to search on, {len(X_final)} for the final test")

1482 samples, 6 features
1185 samples to search on, 291 for the final test


Voici les candidats : les familles de modèles et les réglages qu'un outil d'AutoML essaierait.

In [5]:
candidates = {"Linear": LinearRegression(),
              "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=10.0)),
              "Nearest 3": make_pipeline(StandardScaler(), KNeighborsRegressor(3)),
              "Nearest 30": make_pipeline(StandardScaler(), KNeighborsRegressor(30)),
              "Forest": RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1),
              "Forest, leaf 20": RandomForestRegressor(n_estimators=100, min_samples_leaf=20, random_state=SEED, n_jobs=-1),
              "Boosted": HistGradientBoostingRegressor(random_state=SEED),
              "Boosted, slow": HistGradientBoostingRegressor(learning_rate=0.03, random_state=SEED)}
shuffled = KFold(n_splits=5, shuffle=True, random_state=SEED)
future = TimeSeriesSplit(n_splits=5, gap=GAP)

### À vous (1) : la boucle

Écrivez `mini_automl(candidates, X, y, cv)`. Pour chaque candidat, calculez l'erreur absolue moyenne en validation croisée, puis renvoyez une Series pandas des erreurs triées de la meilleure à la pire.

*Indice :*
```python
scores = {}
for name, model in candidates.items():
    scores[name] = -cross_val_score(model, X, y, cv=cv, scoring="neg_mean_absolute_error").mean()
return pd.Series(scores).sort_values()
```

In [6]:
mini_automl = None   # replace this line with your function: def mini_automl(candidates, X, y, cv): ...

*Solution détaillée ci-dessous (cliquez sur ⋯ pour l'ouvrir).*

In [7]:
def mini_automl_sol(candidates, X, y, cv):
    scores = {}
    for name, model in candidates.items():
        scores[name] = -cross_val_score(model, X, y, cv=cv, scoring="neg_mean_absolute_error").mean()
    return pd.Series(scores).sort_values()


check("mini_automl", globals(), mini_automl_sol, "loop over candidates.items(), score each with cross_val_score, return a sorted Series.",
      measure=lambda f: f({"Linear": LinearRegression(), "Ridge": candidates["Ridge"]}, X_search, y_search, shuffled).to_numpy())

mini_automl: not answered yet. Hint: loop over candidates.items(), score each with cross_val_score, return a sorted Series.
  Using the reference result below.


### À vous (2) : lancer la recherche deux fois

Remplissez `ranking_shuffled` avec votre fonction sur `X_search, y_search` avec `cv=shuffled`, et `ranking_future` avec `cv=future`. Avant de l'exécuter, notez quel modèle vous vous attendez à voir gagner dans chaque cas. L'exécution prend environ 20 secondes.

In [8]:
ranking_shuffled = None   # replace None with your expression
ranking_future = None     # replace None with your expression

*Solution détaillée ci-dessous (cliquez sur ⋯ pour l'ouvrir).*

In [9]:
ranking_shuffled_sol = mini_automl(candidates, X_search, y_search, shuffled)
ranking_future_sol = mini_automl(candidates, X_search, y_search, future)
check("ranking_shuffled", globals(), ranking_shuffled_sol, "mini_automl(candidates, X_search, y_search, shuffled)",
      measure=lambda s: s.sort_index().to_numpy(), tol=0.01)
check("ranking_future", globals(), ranking_future_sol, "mini_automl(candidates, X_search, y_search, future)",
      measure=lambda s: s.sort_index().to_numpy(), tol=0.01)
pd.DataFrame({"shuffled folds (mm)": ranking_shuffled, "future folds (mm)": ranking_future}).sort_values("shuffled folds (mm)").round(2)

ranking_shuffled: not answered yet. Hint: mini_automl(candidates, X_search, y_search, shuffled)
  Using the reference result below.
ranking_future: not answered yet. Hint: mini_automl(candidates, X_search, y_search, future)
  Using the reference result below.


,shuffled folds (mm),future folds (mm)
Forest,0.46,9.58
Boosted,0.58,9.78
Linear,0.77,0.82
Ridge,0.78,0.86
"Forest, leaf 20",0.97,10.40
Nearest 3,1.17,11.30
"Boosted, slow",1.46,10.94
Nearest 30,2.44,13.04


### À vous (3) : le test final, une seule fois

Chaque recherche désigne un gagnant. Ajustez chaque gagnant sur `X_search, y_search` et calculez son erreur absolue moyenne sur `X_final, y_final` : `final_shuffled` pour le gagnant sur plis mélangés, `final_future` pour le gagnant sur plis futurs.

*Indice :* `ranking_shuffled.index[0]` est le nom du gagnant, et `candidates[name]` est le modèle. Puis `mean_absolute_error(y_final, model.fit(X_search, y_search).predict(X_final))`.

In [10]:
final_shuffled = None   # replace None with your expression
final_future = None     # replace None with your expression

*Solution détaillée ci-dessous (cliquez sur ⋯ pour l'ouvrir).*

In [11]:
final_shuffled_sol = mean_absolute_error(y_final, candidates[ranking_shuffled.index[0]].fit(X_search, y_search).predict(X_final))
final_future_sol = mean_absolute_error(y_final, candidates[ranking_future.index[0]].fit(X_search, y_search).predict(X_final))
check("final_shuffled", globals(), final_shuffled_sol, "take the first name in ranking_shuffled, fit that candidate, score it on the final samples.", tol=0.01)
check("final_future", globals(), final_future_sol, "take the first name in ranking_future, fit that candidate, score it on the final samples.", tol=0.01)
print(f"Winner on shuffled folds: {ranking_shuffled.index[0]}, final test {final_shuffled:.2f} mm")
print(f"Winner on future folds:   {ranking_future.index[0]}, final test {final_future:.2f} mm")
print(f"Persistence, final test:  {mean_absolute_error(y_final, X_final[:, FEATURES.index('last')]):.2f} mm")

final_shuffled: not answered yet. Hint: take the first name in ranking_shuffled, fit that candidate, score it on the final samples.
  Using the reference result below.
final_future: not answered yet. Hint: take the first name in ranking_future, fit that candidate, score it on the final samples.
  Using the reference result below.
Winner on shuffled folds: Forest, final test 13.36 mm
Winner on future folds:   Linear, final test 0.74 mm
Persistence, final test:  1.12 mm


**Pause et discussion.**

- La même boucle, les mêmes candidats et les mêmes données ont choisi deux modèles différents. Qu'est-ce qui a changé dans la boucle ?
- Lee et al. (2023) ont trouvé l'AutoML devant avec un échantillonnage aléatoire et à égalité lorsque des années entières étaient tenues à l'écart. Quelle comparaison correspond à l'humidité du sol journalière de l'année prochaine ?
- Que demanderiez-vous à un article qui rapporte « le meilleur de 20 modèles sélectionnés par AutoML » ?

:::{dropdown} Points à faire ressortir
Seule la validation a changé. Une recherche récompense ce que le score récompense, et les plis mélangés récompensent la copie des voisins, ce que les forêts font bien. Demandez ensuite à l'article : qu'est-ce qui a été tenu à l'écart, quel modèle de référence (*baseline*) le gagnant a-t-il battu, et le score rapporté a-t-il servi à choisir le gagnant ? Les paquets font gagner du temps sur la boucle. Ils ne savent pas comment votre modèle sera utilisé.
:::

## Partie 2. Chercher des réglages, et pourquoi le meilleur score est optimiste

La boucle s'étend aux réglages : tirer un modèle au hasard, le noter, garder le meilleur. `GridSearchCV` et `RandomizedSearchCV` de scikit-learn font la même chose avec moins de saisie. Des bibliothèques comme Optuna tirent de nouveaux réglages près de ceux qui ont obtenu de bons scores, ce qui fait gagner du temps quand chaque ajustement prend des heures, et elles optimisent le même score.

Plus une recherche essaie de candidats, plus il est probable que son meilleur score tienne en partie à la chance. Pour le rendre visible, cette recherche n'utilise que 300 des événements sismiques d'entraînement de [3.5](3.5_multiclass_classification.ipynb), et elle enregistre aussi le score de test de chaque candidat, ce que nous ne ferions jamais dans une vraie recherche. Elle prend environ 30 secondes.

In [12]:
SEISMIC_FILES = {
    "1000_earthquakes_physical_features.csv": "md5:28129c8dd1b3e14f655d489577b841b5",
    "1000_explosion_physical_features.csv": "md5:af1342d32e163e961e043364136359b0",
    "1000_noise_physical_features.csv": "md5:16cdb992fed6cf6273d5624f5df905da",
    "1000_surface_physical_features.csv": "md5:9a2c2643030cf058704d68e130654e9d",
}
try:
    tables = [pd.read_csv(pooch.retrieve(url=f"https://zenodo.org/api/records/14025693/files/{name}/content",
                                         known_hash=checksum, fname=name, path=pooch.os_cache("mlgeo")), index_col=0)
              for name, checksum in SEISMIC_FILES.items()]
    seismic = pd.concat(tables, ignore_index=True).dropna(axis=1)
    X = seismic.drop(columns=["source", "serial_no"])
    y = seismic["source"]
except Exception as error:
    print(f"Zenodo could not be reached ({error}). Using SYNTHETIC events: the scores below will differ from the text.")
    values, codes = make_classification(n_samples=4000, n_features=61, n_informative=20, n_classes=4, random_state=SEED)
    X = pd.DataFrame(values, columns=[f"feature_{i}" for i in range(61)])
    y = pd.Series(np.array(["earthquake", "explosion", "noise", "surface event"])[codes])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SPLIT_SEED, stratify=y)
truth = y_test.to_numpy()
print(f"{len(X_train)} training events, {len(X_test)} test events, {X.shape[1]} waveform features")

3000 training events, 1000 test events, 61 waveform features


In [13]:
macro = lambda a, b: f1_score(a, b, average="macro")
few = np.random.default_rng(1).choice(len(X_train), 300, replace=False)
X_few, y_few = X_train.iloc[few], y_train.iloc[few]
draw = np.random.default_rng(SEED)
tried = []
for _ in range(30):
    kind = draw.choice(["logistic", "forest", "nearest"])
    if kind == "logistic":
        model = make_pipeline(StandardScaler(), LogisticRegression(C=10 ** draw.uniform(-3, 2), max_iter=5000))
    elif kind == "forest":
        model = RandomForestClassifier(n_estimators=100, min_samples_leaf=int(draw.choice([1, 2, 5, 10, 20])),
                                       max_features=float(draw.choice([0.1, 0.33, 1.0])), random_state=SEED, n_jobs=-1)
    else:
        model = make_pipeline(StandardScaler(), KNeighborsClassifier(int(draw.choice([1, 3, 5, 10, 20, 40]))))
    tried.append({"kind": kind, "cv": cross_val_score(model, X_few, y_few, cv=5, scoring="f1_macro").mean(),
                  "test": macro(y_test, model.fit(X_few, y_few).predict(X_test))})
tried = pd.DataFrame(tried).sort_values("cv", ascending=False)
tried.head(5).round(3)

,kind,cv,test
21,forest,0.845,0.820
7,forest,0.826,0.809
12,forest,0.818,0.810
27,forest,0.818,0.810
8,forest,0.817,0.813


**Lisez le tableau.** Comparez la valeur `cv` la plus haute à la valeur `test` de la même ligne. L'écart est l'optimisme de la recherche, et la ligne du haut n'est souvent pas la meilleure en test. Rapportez le score de test du gagnant, mesuré une seule fois, et jamais le meilleur score de la recherche.

## Partie 3. Auditer un script écrit par un agent

Aujourd'hui, une grande partie de la recherche est écrite par un agent de code plutôt que par un paquet. Voici le genre de script qu'un agent produira si on lui demande de « trouver le meilleur modèle pour cette table sismique ». Il s'exécute, et il affiche un score élevé.

In [14]:
# --- AI-generated modeling script: do NOT trust it yet ---
two_labels = seismic[seismic["source"] != "surface event"]   # remove noisy labels
features = StandardScaler().fit_transform(two_labels.drop(columns=["source", "serial_no"]))   # normalize features
a_train, a_test, b_train, b_test = train_test_split(features, two_labels["source"], test_size=0.25, random_state=7)

best_name, best_score = None, -1.0
for name, model in {"logistic_regression": LogisticRegression(max_iter=5000),
                    "random_forest": RandomForestClassifier(n_estimators=100, random_state=7, n_jobs=-1)}.items():
    model.fit(a_train, b_train)
    score = model.score(a_train, b_train)   # evaluate each candidate
    print(f"{name}: accuracy = {score:.3f}")
    if score > best_score:
        best_name, best_score = name, score
print(f"best model: {best_name}, accuracy = {best_score:.3f}")

logistic_regression: accuracy = 0.915
random_forest: accuracy = 1.000
best model: random_forest, accuracy = 1.000


```{admonition} Tâche : auditer le script
:class: attention
Il comporte trois défauts distincts. Notez-les et, pour chacun, dites ce qu'il fait au nombre rapporté, avant d'ouvrir la réponse.
```

:::{dropdown} Les trois défauts
1. **Une classe est supprimée.** `seismic["source"] != "surface event"` retire l'un des quatre types de sources, sous un commentaire sur des étiquettes bruitées. Le score décrit désormais un problème plus facile, et le modèle ne pourra jamais prédire un événement de surface.
2. **Le *scaler* est ajusté avant le découpage.** `fit_transform` sur tous les échantillons utilise les événements de test pour calculer les moyennes et les variances. L'effet est faible ici, mais avec une imputation ou un encodage de la cible il peut être important.
3. **Les modèles sont choisis sur l'exactitude d'entraînement.** `model.score(a_train, b_train)` récompense la mémorisation : une forêt atteint 1,000 sur des données qu'elle a vues.

Signes d'alerte : un score presque parfait, aucun modèle de référence, aucun score sur des données tenues à l'écart.
:::

### À vous (4) : le réparer

Placez le *scaler* dans un *pipeline* scikit-learn et utilisez le découpage canonique avec les quatre classes. Calculez `fair_cv`, le macro-F1 de la forêt en validation croisée à 5 plis sur `X_train, y_train`. Puis calculez `fair_test`, son macro-F1 sur les événements de test, une seule fois.

*Indice :* `pipeline = make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=100, random_state=7, n_jobs=-1))`, puis `cross_val_score(pipeline, X_train, y_train, cv=5, scoring="f1_macro").mean()`, puis `macro(y_test, pipeline.fit(X_train, y_train).predict(X_test))`.

In [15]:
fair_cv = None     # replace None with your expression
fair_test = None   # replace None with your expression

*Solution détaillée ci-dessous (cliquez sur ⋯ pour l'ouvrir).*

In [16]:
pipeline_sol = make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=100, random_state=7, n_jobs=-1))
fair_cv_sol = cross_val_score(pipeline_sol, X_train, y_train, cv=5, scoring="f1_macro").mean()
fair_test_sol = macro(y_test, pipeline_sol.fit(X_train, y_train).predict(X_test))
check("fair_cv", globals(), fair_cv_sol, "use cross_val_score on X_train, y_train with scoring='f1_macro'.", tol=0.01)
check("fair_test", globals(), fair_test_sol, "fit the pipeline on all of X_train, then score on X_test.", tol=0.01)
print(f"Agent's script: accuracy 1.000 on three classes, on data it had seen")
print(f"Repaired: macro-F1 {fair_cv:.3f} in cross-validation, {fair_test:.3f} on test, four classes (always guessing one class: 0.10)")

fair_cv: not answered yet. Hint: use cross_val_score on X_train, y_train with scoring='f1_macro'.
  Using the reference result below.
fair_test: not answered yet. Hint: fit the pipeline on all of X_train, then score on X_test.
  Using the reference result below.
Agent's script: accuracy 1.000 on three classes, on data it had seen
Repaired: macro-F1 0.871 in cross-validation, 0.888 on test, four classes (always guessing one class: 0.10)


**Si vous avez le temps : essayez avec votre propre agent.** Demandez à un agent de code un court AutoML utilisant scikit-learn seul, sur les données de votre projet ou sur P395. Vérifiez ensuite sa réponse à l'aide de cette liste : que tient-il à l'écart, quel modèle de référence rapporte-t-il, le *scaler* est-il ajusté uniquement sur les plis d'entraînement, l'ensemble de test est-il utilisé une seule fois ? Apportez ce que vous trouvez à la séance de vendredi sur l'évaluation des agents ([6.3](../Chapter6-AgenticAI/6.3_build_an_eval_set.ipynb)).

## À retenir

L'automatisation est passée des paquets de recherche aux agents qui écrivent toute la chaîne de modélisation à la demande. Les vérifications sont restées les mêmes : un modèle de référence, un découpage qui correspond à l'usage du modèle, une validation croisée à l'intérieur des données d'entraînement et un seul regard sur un ensemble de test tenu à l'écart. Automatisez la recherche, et vérifiez vous-même le résultat.